In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🏅 Medallion Architecture Mock — Bronze / Silver / Gold
# MAGIC
# MAGIC Gera **5 milhões de linhas** de dados de vendas (e-commerce) e processa
# MAGIC em três camadas usando **PySpark + Delta Lake**.
# MAGIC
# MAGIC - **Bronze** → dados crus, "sujos" (nulos, duplicados, tipos errados, datas em texto)
# MAGIC - **Silver** → dados limpos, tipados, deduplicados e validados
# MAGIC - **Gold** → tabelas agregadas prontas para BI / análise
# MAGIC
# MAGIC Feito para rodar no **Databricks Free Edition** (serverless + Unity Catalog).

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Configuração — catálogo e schema

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql import types as T

# No Free Edition o catálogo padrão é "workspace".
CATALOG = "workspace"
SCHEMA  = "medallion_demo"
N_ROWS  = 5_000_000   # 5 milhões

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Usando {CATALOG}.{SCHEMA} | gerando {N_ROWS:,} linhas")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. 🥈 SILVER — limpeza, tipagem, deduplicação e validação
# MAGIC
# MAGIC - Remove duplicatas por `transaction_id`
# MAGIC - Converte a data string → `timestamp`
# MAGIC - Padroniza categoria (capitalização única)
# MAGIC - Filtra registros inválidos (preço/quantidade ≤ 0)
# MAGIC - Calcula `total_amount`

# COMMAND ----------

from pyspark.sql.window import Window

bronze = spark.table("bronze_sales")

w = Window.partitionBy("transaction_id").orderBy(F.col("_ingest_ts").desc())

silver_df = (
    bronze
    # dedupe: mantém o registro mais recente por transaction_id
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    # tipagem da data
    .withColumn("event_ts", F.to_timestamp("event_ts"))
    .withColumn("event_date", F.to_date("event_ts"))
    # padroniza categoria -> Initcap
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    # validações de qualidade
    .filter((F.col("unit_price") > 0) & (F.col("quantity") > 0))
    .filter(F.col("event_ts").isNotNull())
    # métrica derivada
    .withColumn("total_amount", F.round(F.col("unit_price") * F.col("quantity"), 2))
    # trata email nulo
    .withColumn("email", F.coalesce(F.col("email"), F.lit("desconhecido")))
    .select(
        "transaction_id", "customer_id", "product_id", "category",
        "country", "payment_method", "unit_price", "quantity",
        "total_amount", "email", "event_ts", "event_date"
    )
)

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_sales"))

#total = spark.table("silver_sales").count()
#print(f"Silver gravada: {total:,} linhas (após dedupe + validação)")
#display(spark.table("silver_sales").limit(10))
